In [ ]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import time

# Initialize client
client = OpenAI()

# Load human-coded data (300 cases only)
df = pd.read_csv("human_coded_sample_300.csv")

# Category whitelist
VALID_LABELS = {
    "Economy",
    "Healthcare",
    "Politics/Democracy",
    "Social Issues",
    "Crime",
    "Immigration",
    "Environment",
    "Foreign Policy",
    "Other",
    "Unclear"
}

# Prompt template
PROMPT_TEMPLATE = """
You are a survey research assistant coding open-ended responses to a public opinion question:

“Mention one most important problem facing the country.”

Your task is to assign EACH response to exactly ONE of the following categories:

- Economy
- Healthcare
- Politics/Democracy
- Social Issues
- Crime
- Immigration
- Environment
- Foreign Policy
- Other
- Unclear

DEFINITION AND DECISION RULES:

1. Assign only ONE category per response.
2. If multiple issues are mentioned, code the FIRST or most emphasized issue.
3. If a response criticizes government, leaders, institutions, elections, democracy, corruption, polarization, or partisanship without naming a specific policy area, code as Politics/Democracy.
4. Responses referencing social division, polarization, cultural conflict, racial tension, or lack of social cohesion without explicit reference to political institutions should be coded as Social Issues.
5. Economic issues include inflation, prices, wages, jobs, debt, cost of living, taxes, and economic inequality.
6. Healthcare includes access, affordability, insurance, hospitals, and public health.
7. Immigration includes borders, illegal immigration, asylum, migrants, or refugee policy.
8. Crime includes violence, drugs, law enforcement, public safety, and incarceration.
9. Environment includes climate change, pollution, environmental protection, and natural disasters.
10. Foreign Policy includes wars, international relations, foreign governments, and national security abroad.
11. Responses that are vague, uninterpretable, or say “don’t know” should be coded as Unclear.
12. Responses that do not clearly fit any category should be coded as Other.

OUTPUT RULES:
- Output ONLY the category name.
- Do NOT explain your reasoning.
- Output must exactly match one of the category labels.

Response to code:
"{response_text}"
""".strip()

# Function to classify one response
def classify_response(text):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        temperature=0,
        messages=[
            {"role": "user", "content": PROMPT_TEMPLATE.format(response_text=text)}
        ],
    )
    
    label = response.choices[0].message.content.strip()

    if label not in VALID_LABELS:
        return "INVALID_OUTPUT"
    
    return label

# Run LLM coding
llm_codes = []

for text in tqdm(df["response_text"]):
    label = classify_response(text)
    llm_codes.append(label)
    time.sleep(0.2)  # rate-limit safety

# Attach results
df["llm_code"] = llm_codes

# Save output
df.to_csv("llm_coded_sample_300.csv", index=False)

print("LLM coding complete. Results saved to llm_coded_sample.csv")


100%|██████████| 300/300 [03:27<00:00,  1.45it/s]

LLM coding complete. Results saved to llm_coded_sample.csv


In [21]:
# make sure there are not blanks or invalid categorizations
df["llm_code"].value_counts()


llm_code
Economy               94
Social Issues         56
Immigration           52
Politics/Democracy    52
Unclear               13
Healthcare            11
Crime                  7
Other                  6
Environment            5
Foreign Policy         4
Name: count, dtype: int64

In [22]:
#check the percent match of the human coded vs llm coded
percent_match = (df["human_code_label"] == df["llm_code"]).mean()
print(percent_match)

0.9266666666666666


In [23]:
# create confusion matrix to show where diagreements are 
from sklearn.metrics import confusion_matrix
import pandas as pd

labels = [
    "Economy",
    "Healthcare",
    "Politics/Democracy",
    "Social Issues",
    "Crime",
    "Immigration",
    "Environment",
    "Foreign Policy",
    "Other",
    "Unclear"
]

cm = confusion_matrix(
    df["human_code_label"],
    df["llm_code"],
    labels=labels
)

cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df


,Economy,Healthcare,Politics/Democracy,Social Issues,Crime,Immigration,Environment,Foreign Policy,Other,Unclear
Economy,87,0,0,1,0,2,0,0,1,0
Healthcare,0,11,0,0,0,0,0,0,0,0
Politics/Democracy,4,0,48,0,0,0,0,0,1,1
Social Issues,2,0,1,54,0,0,0,0,0,0
Crime,0,0,0,1,6,0,0,0,0,0
Immigration,1,0,0,0,0,50,0,0,0,1
Environment,0,0,0,0,0,0,5,0,0,0
Foreign Policy,0,0,0,0,1,0,0,4,0,0
Other,0,0,0,0,0,0,0,0,3,1
Unclear,0,0,3,0,0,0,0,0,1,10


In [24]:
#create classification report
from sklearn.metrics import classification_report

print(classification_report(
    df["human_code_label"],
    df["llm_code"],
    labels=labels,
    zero_division=0
))


                    precision    recall  f1-score   support

           Economy       0.93      0.96      0.94        91
        Healthcare       1.00      1.00      1.00        11
Politics/Democracy       0.92      0.89      0.91        54
     Social Issues       0.96      0.95      0.96        57
             Crime       0.86      0.86      0.86         7
       Immigration       0.96      0.96      0.96        52
       Environment       1.00      1.00      1.00         5
    Foreign Policy       1.00      0.80      0.89         5
             Other       0.50      0.75      0.60         4
           Unclear       0.77      0.71      0.74        14

          accuracy                           0.93       300
         macro avg       0.89      0.89      0.89       300
      weighted avg       0.93      0.93      0.93       300



In [25]:
#pull disagreements to compare

errors = df[df["human_code_label"] != df["llm_code"]]
errors.head(10)


,respondent_id,response_text,human_code,human_code_label,notes,llm_code
1,141685,don't know what to write,9,Other,NaN,Unclear
14,220922,"education, inflation, peace, gun reform, trump...",4,Social Issues,multiple problems,Economy
30,140474,project 2025 Donald Trump,3,Politics/Democracy,NaN,Unclear
32,141810,"inflation, illegal immigration, endless suppor...",3,Politics/Democracy,multiple problems,Economy
42,201019,Fundamental Human Rights Values,3,Politics/Democracy,NaN,Other
49,235273,Spending of monies,1,Economy,NaN,Other
54,345378,security- dont feel is as strong as we used to...,8,Foreign Policy,NaN,Crime
68,233683,"immigration, high rent , high Mirra gage expen...",1,Economy,multiple problems,Immigration
84,319050,Government spending. Is the cause and reason w...,3,Politics/Democracy,NaN,Economy
102,142428,lack of integrity,10,Unclear,NaN,Politics/Democracy


In [ ]:
errors.value_counts(["human_code_label", "llm_code"]).head(10)
errors.to_csv("audit_disagreements_sample"
".csv", index=False)